# Devoluciones — Data Cleaning & Normalization

Analyze the `database-devolutions-sic-jac.csv` reference data from SIC JAC.
Focus on `Seccion`, `Descripcion`, `Unidad`, and the balance/entry/exit quantities.

**Source:** `docs/reference/warehouse/raw/database-devolutions-sic-jac.csv`

In [1]:
import pandas as pd
from pathlib import Path

SRC = Path('../../reference/warehouse/raw/database-devolutions-sic-jac.csv')

In [2]:
df = pd.read_csv(SRC, sep=',', encoding='utf-8', low_memory=False)
print(f'Shape: {df.shape}')
df.head(10)

Shape: (19, 11)


,Seccion,Descripcion,Unidad,Saldo_31122024_Cantidad,Saldo_31122024_Valor,Entrada_Cantidad,Entrada_Valor,Salida_Cantidad,Salida_Valor,Saldos_Cantidad,Saldos_Valor
0,A 1 - TITULO 2/18,CONFITE 2003 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0
1,A 1 - TITULO 2/18,ROJO 3006 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0
2,A 1 - TITULO 2/18,VERDE MEDIO 4029 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0
3,A 1 - TITULO 2/18,AZUL PETROLEO 7084 2/18 DEV,KILOS,0.0,0.0,204.0,204.0,0.0,0.0,204.0,204.0
4,A 1 - TITULO 2/18,CLAVEL 3005 2/18,KILOS,11.0,11.0,0.0,0.0,11.0,11.0,0.0,0.0
5,A 1 - TITULO 2/18,VERDE BOTELLA 4011 2/18,KILOS,18.0,18.0,0.0,0.0,18.0,18.0,0.0,0.0
6,A 1 - TITULO 2/18,VERDE LECHUGA 4007 2/18 DEV,KILOS,0.0,0.0,203.0,203.0,0.0,0.0,203.0,203.0
7,A 1 - TITULO 2/18,CICLAN BAJO 2011 2/18,KILOS,0.5,0.5,0.0,0.0,0.5,0.5,0.0,0.0
8,A 1 - TITULO 2/18,AMARILLO CANARIO 0015 2/18,KILOS,0.5,0.5,0.0,0.0,0.5,0.5,0.0,0.0
9,A 1 - TITULO 2/18,CONFITE 2008 2/18,KILOS,0.0,0.0,204.0,204.0,0.0,0.0,204.0,204.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Seccion                  19 non-null     str    
 1   Descripcion              19 non-null     str    
 2   Unidad                   19 non-null     str    
 3   Saldo_31122024_Cantidad  19 non-null     float64
 4   Saldo_31122024_Valor     19 non-null     float64
 5   Entrada_Cantidad         19 non-null     float64
 6   Entrada_Valor            19 non-null     float64
 7   Salida_Cantidad          19 non-null     float64
 8   Salida_Valor             19 non-null     float64
 9   Saldos_Cantidad          19 non-null     float64
 10  Saldos_Valor             19 non-null     float64
dtypes: float64(8), str(3)
memory usage: 1.8 KB


In [4]:
df.isnull().sum()

Seccion                    0
Descripcion                0
Unidad                     0
Saldo_31122024_Cantidad    0
Saldo_31122024_Valor       0
Entrada_Cantidad           0
Entrada_Valor              0
Salida_Cantidad            0
Salida_Valor               0
Saldos_Cantidad            0
Saldos_Valor               0
dtype: int64

---
## 1. Column overview

`Descripcion` appears to contain product names, color codes, thickness, and the `DEV` marker for devolutions. We need to normalize these strings and verify that `Saldo`, `Entrada`, `Salida`, and `Saldos` fields behave as expected.

In [5]:
df.columns.tolist()

['Seccion',
 'Descripcion',
 'Unidad',
 'Saldo_31122024_Cantidad',
 'Saldo_31122024_Valor',
 'Entrada_Cantidad',
 'Entrada_Valor',
 'Salida_Cantidad',
 'Salida_Valor',
 'Saldos_Cantidad',
 'Saldos_Valor']

In [6]:
cols = [c.strip() for c in df.columns]
df.columns = cols

df[['Seccion','Descripcion','Unidad']].head(10)

,Seccion,Descripcion,Unidad
0,A 1 - TITULO 2/18,CONFITE 2003 2/18,KILOS
1,A 1 - TITULO 2/18,ROJO 3006 2/18,KILOS
2,A 1 - TITULO 2/18,VERDE MEDIO 4029 2/18,KILOS
3,A 1 - TITULO 2/18,AZUL PETROLEO 7084 2/18 DEV,KILOS
4,A 1 - TITULO 2/18,CLAVEL 3005 2/18,KILOS
5,A 1 - TITULO 2/18,VERDE BOTELLA 4011 2/18,KILOS
6,A 1 - TITULO 2/18,VERDE LECHUGA 4007 2/18 DEV,KILOS
7,A 1 - TITULO 2/18,CICLAN BAJO 2011 2/18,KILOS
8,A 1 - TITULO 2/18,AMARILLO CANARIO 0015 2/18,KILOS
9,A 1 - TITULO 2/18,CONFITE 2008 2/18,KILOS


---
## 2. Inspect `Descripcion`

Expected pattern examples:
- `CONFITE 2003 2/18`
- `AZUL PETROLEO 7084 2/18 DEV`
- `AMARILLO CANARIO 0015 2/18`

The goal is to split into `product_name`, `color_code`, `thickness`, and `is_devolucion`.

In [7]:
desc = df['Descripcion'].astype(str)
print(f'Unique descriptions: {desc.nunique()} / {len(desc)} rows')
print(f'Null descriptions: {desc.isnull().sum()}')
print('\nSample descriptions:')
for value in desc.dropna().head(20).tolist():
    print(' -', value)

Unique descriptions: 19 / 19 rows
Null descriptions: 0

Sample descriptions:
 - CONFITE 2003 2/18
 - ROJO 3006 2/18
 - VERDE MEDIO 4029 2/18
 - AZUL PETROLEO 7084 2/18 DEV
 - CLAVEL 3005 2/18
 - VERDE BOTELLA 4011 2/18
 - VERDE LECHUGA 4007 2/18 DEV
 - CICLAN BAJO 2011 2/18
 - AMARILLO CANARIO 0015 2/18
 - CONFITE 2008 2/18
 - CICLAN 2004 2/18 DEV
 - MIXTOS 2/18
 - VERDE 4026 2/32
 - KANTUTA 3045 2/32-D DEV
 - BLANCO 0010 4/9
 - CELESTE 7054 2/12 DEV
 - AZUL FRANCIA 7007 3/15 MADEJITAS
 - VERDE LECHUGA 4007 2/13 DEV
 - NEGRO 5000 2/13-N DEV


In [8]:
import re

pattern = re.compile(r'^(?P<product>.+?)\s+(?P<color_code>\d{3,4})\s+(?P<thickness>\d+/\d+)(?:\s+(?P<suffix>DEV|DEVOL|DEVOLUCIONES|DEVOLU|DEV.*?))?$', re.IGNORECASE)

matched = desc.str.contains(pattern, na=False)
print('Matched rows:', matched.sum())
print('Total rows:', len(desc))

Matched rows: 15
Total rows: 19


C:\Users\Alexander\AppData\Local\Temp\ipykernel_11308\2876954922.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matched = desc.str.contains(pattern, na=False)


In [9]:
unmatched = df[~desc.str.contains(pattern, na=False)]
print('Unmatched sample count:', len(unmatched))
unmatched[['Descripcion']].head(20)

Unmatched sample count: 4


C:\Users\Alexander\AppData\Local\Temp\ipykernel_11308\2464270272.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  unmatched = df[~desc.str.contains(pattern, na=False)]


,Descripcion
11,MIXTOS 2/18
13,KANTUTA 3045 2/32-D DEV
16,AZUL FRANCIA 7007 3/15 MADEJITAS
18,NEGRO 5000 2/13-N DEV


---
## 3. Parse `Descripcion`

Separate the description into `product_name`, `color_code`, `thickness`, and `dev_flag`. This helps standardize `DEV` as a normalized devolucion marker.

In [10]:
def parse_devolucion_desc(description):
    if not isinstance(description, str) or not description.strip():
        return pd.Series({
            'product_name': None,
            'color_code': None,
            'thickness': None,
            'dev_flag': False,
            'raw_suffix': None,
        })

    text = description.strip()
    dev_flag = bool(re.search(r'\bDEV\b|\bDEVOL\w*|\bDEVOLU\w*', text, re.IGNORECASE))
    text = re.sub(r'\bDEV\b|\bDEVOL\w*|\bDEVOLU\w*', '', text, flags=re.IGNORECASE).strip()

    match = pattern.match(text)
    if match:
        return pd.Series({
            'product_name': match.group('product').strip(),
            'color_code': match.group('color_code').strip(),
            'thickness': match.group('thickness').strip(),
            'dev_flag': dev_flag,
            'raw_suffix': None,
        })

    parts = text.split()
    product_name = None
    color_code = None
    thickness = None
    if len(parts) >= 3 and re.match(r'^\d{3,4}$', parts[-2]) and re.match(r'^\d+/\d+$', parts[-1]):
        product_name = ' '.join(parts[:-2])
        color_code = parts[-2]
        thickness = parts[-1]
    elif len(parts) >= 2 and re.match(r'^\d+/\d+$', parts[-1]):
        product_name = ' '.join(parts[:-1])
        thickness = parts[-1]
    else:
        product_name = text

    return pd.Series({
        'product_name': product_name,
        'color_code': color_code,
        'thickness': thickness,
        'dev_flag': dev_flag,
        'raw_suffix': None,
    })

parsed_desc = df['Descripcion'].apply(parse_devolucion_desc)
df = pd.concat([df, parsed_desc], axis=1)
df[['Descripcion','product_name','color_code','thickness','dev_flag']].head(15)

,Descripcion,product_name,color_code,thickness,dev_flag
0,CONFITE 2003 2/18,CONFITE,2003,2/18,False
1,ROJO 3006 2/18,ROJO,3006,2/18,False
2,VERDE MEDIO 4029 2/18,VERDE MEDIO,4029,2/18,False
3,AZUL PETROLEO 7084 2/18 DEV,AZUL PETROLEO,7084,2/18,True
4,CLAVEL 3005 2/18,CLAVEL,3005,2/18,False
5,VERDE BOTELLA 4011 2/18,VERDE BOTELLA,4011,2/18,False
6,VERDE LECHUGA 4007 2/18 DEV,VERDE LECHUGA,4007,2/18,True
7,CICLAN BAJO 2011 2/18,CICLAN BAJO,2011,2/18,False
8,AMARILLO CANARIO 0015 2/18,AMARILLO CANARIO,0015,2/18,False
9,CONFITE 2008 2/18,CONFITE,2008,2/18,False


In [11]:
print('Parsed totals:')
print('dev_flag count:', df['dev_flag'].sum())
print('Missing color_code:', df['color_code'].isnull().sum())
print('Missing thickness:', df['thickness'].isnull().sum())

Parsed totals:
dev_flag count: 7
Missing color_code: 4
Missing thickness: 3


In [12]:
unparsed = df[df['color_code'].isnull() | df['thickness'].isnull()]
unparsed[['Descripcion','product_name','color_code','thickness','dev_flag']].head(30)

,Descripcion,product_name,color_code,thickness,dev_flag
11,MIXTOS 2/18,MIXTOS,NaN,2/18,False
13,KANTUTA 3045 2/32-D DEV,KANTUTA 3045 2/32-D,NaN,NaN,True
16,AZUL FRANCIA 7007 3/15 MADEJITAS,AZUL FRANCIA 7007 3/15 MADEJITAS,NaN,NaN,False
18,NEGRO 5000 2/13-N DEV,NEGRO 5000 2/13-N,NaN,NaN,True


---
## 4. Validate quantity columns

Confirm that `Saldo`, `Entrada`, `Salida`, and `Saldos` behave as expected, and check for negative or unexpected values.

In [13]:
for col in ['Saldo_31122024_Cantidad','Saldo_31122024_Valor','Entrada_Cantidad','Entrada_Valor','Salida_Cantidad','Salida_Valor','Saldos_Cantidad','Saldos_Valor']:
    if col in df.columns:
        print(col, 'min', df[col].min(), 'max', df[col].max(), 'nulls', df[col].isnull().sum())
    else:
        print(col, 'missing')

Saldo_31122024_Cantidad min 0.0 max 18.0 nulls 0
Saldo_31122024_Valor min 0.0 max 18.0 nulls 0
Entrada_Cantidad min 0.0 max 204.0 nulls 0
Entrada_Valor min 0.0 max 204.0 nulls 0
Salida_Cantidad min 0.0 max 204.0 nulls 0
Salida_Valor min 0.0 max 204.0 nulls 0
Saldos_Cantidad min 0.0 max 204.0 nulls 0
Saldos_Valor min 0.0 max 204.0 nulls 0


In [14]:
if 'Entrada_Cantidad' in df.columns and 'Salida_Cantidad' in df.columns and 'Saldos_Cantidad' in df.columns:
    inconsistent = df[~((df['Saldos_Cantidad'] == df['Saldo_31122024_Cantidad']) | (df['Saldos_Cantidad'] == df['Entrada_Cantidad'] - df['Salida_Cantidad']))] 
    print('Potential balance mismatches:', len(inconsistent))
    inconsistent.head(10)
else:
    print('Required quantity columns not all present')

Potential balance mismatches: 6


---
## 5. Proposal: Normalized warehouse schema

| Column | Source | Notes |
|---|---|---|
| `seccion` | `Seccion` | product category / title row grouping |
| `product_name` | parsed `Descripcion` | normalized product description |
| `color_code` | parsed `Descripcion` | numeric color code |
| `thickness` | parsed `Descripcion` | yarn thickness or gauge |
| `is_devolucion` | parsed `Descripcion` | true when description includes devolucion marker |
| `unidad` | `Unidad` | measure unit, e.g. KILOS |
| `saldo_cantidad` | `Saldo_31122024_Cantidad` | closing quantity as of 31-12-2024 |
| `saldo_valor` | `Saldo_31122024_Valor` | closing value |
| `entrada_cantidad` | `Entrada_Cantidad` | quantity entered |
| `salida_cantidad` | `Salida_Cantidad` | quantity removed |
| `saldos_cantidad` | `Saldos_Cantidad` | current balance quantity |

In [15]:
cleaned = df.copy()
cleaned = cleaned.rename(columns={
    'Saldo_31122024_Cantidad': 'saldo_cantidad',
    'Saldo_31122024_Valor': 'saldo_valor',
    'Entrada_Cantidad': 'entrada_cantidad',
    'Entrada_Valor': 'entrada_valor',
    'Salida_Cantidad': 'salida_cantidad',
    'Salida_Valor': 'salida_valor',
    'Saldos_Cantidad': 'saldos_cantidad',
    'Saldos_Valor': 'saldos_valor',
    'Unidad': 'unidad',
    'Seccion': 'seccion',
})
cleaned[['seccion','product_name','color_code','thickness','dev_flag','unidad','saldo_cantidad','entrada_cantidad','salida_cantidad','saldos_cantidad']].head(15)

,seccion,product_name,color_code,thickness,dev_flag,unidad,saldo_cantidad,entrada_cantidad,salida_cantidad,saldos_cantidad
0,A 1 - TITULO 2/18,CONFITE,2003,2/18,False,KILOS,0.0,204.0,204.0,0.0
1,A 1 - TITULO 2/18,ROJO,3006,2/18,False,KILOS,0.0,204.0,204.0,0.0
2,A 1 - TITULO 2/18,VERDE MEDIO,4029,2/18,False,KILOS,0.0,204.0,204.0,0.0
3,A 1 - TITULO 2/18,AZUL PETROLEO,7084,2/18,True,KILOS,0.0,204.0,0.0,204.0
4,A 1 - TITULO 2/18,CLAVEL,3005,2/18,False,KILOS,11.0,0.0,11.0,0.0
5,A 1 - TITULO 2/18,VERDE BOTELLA,4011,2/18,False,KILOS,18.0,0.0,18.0,0.0
6,A 1 - TITULO 2/18,VERDE LECHUGA,4007,2/18,True,KILOS,0.0,203.0,0.0,203.0
7,A 1 - TITULO 2/18,CICLAN BAJO,2011,2/18,False,KILOS,0.5,0.0,0.5,0.0
8,A 1 - TITULO 2/18,AMARILLO CANARIO,0015,2/18,False,KILOS,0.5,0.0,0.5,0.0
9,A 1 - TITULO 2/18,CONFITE,2008,2/18,False,KILOS,0.0,204.0,0.0,204.0


---
## 6. Next steps

- Refine parser for descriptions with extra suffixes and nonstandard tokens.
- Verify that `DEV` entries are true devolutions and not label noise.
- Compare `color_code` and `thickness` against other warehouse datasets if available.
- Export cleaned output to a CSV or reference dataset.

In [16]:
cleaned_path = Path('devolution-cleaning-normalized.csv')
cleaned.to_csv(cleaned_path, index=False)
print('Exported normalized data to', cleaned_path)

Exported normalized data to devolution-cleaning-normalized.csv
